# 2.01 OpenAI Embeddings

OpenAI의 `text-embedding-3-small` / `text-embedding-3-large` 모델과 Azure OpenAI 배포판을 다룬다.
RAG, 의미 검색, 클러스터링의 기본값으로 가장 널리 쓰이며, **차원 축소(`dimensions`)** 와 **배치 처리(`chunk_size`)** 가 이 공급자의 실무 포인트다.

## 학습 목표

- `OpenAIEmbeddings`로 단일 질의(`embed_query`)와 문서 묶음(`embed_documents`)을 처리한다
- `text-embedding-3-small` (1536d) vs `-large` (3072d)의 차원·비용 트레이드오프를 이해한다
- `dimensions` 축소 파라미터로 vector store 스키마를 고정한다
- Azure OpenAI 배포(`AzureOpenAIEmbeddings`)의 필수 필드 4종을 안다

## 언제 쓰나

- **다국어 RAG의 기본값**: 한국어 포함 대부분 언어에서 안정적 품질
- **차원 축소가 필요한 스토리지 최적화**: 1536 → 512로 줄여서 저장/연산 비용 ↓
- **엔터프라이즈**: Azure 배포로 지역·규정 준수 요구 충족

## 2.01.1 환경 설정

필요 패키지: `langchain-openai`. `.env`에 `OPENAI_API_KEY`가 있어야 한다. Azure를 쓴다면 `AZURE_OPENAI_API_KEY`, `AZURE_OPENAI_ENDPOINT`도 함께 설정.

In [ ]:
# !pip install -U langchain langchain-openai

import os
from dotenv import load_dotenv
load_dotenv(override=True)

assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY 필요"

## 2.01.2 기본 사용법

`embed_query(text: str) -> list[float]`는 **검색 쿼리**에, `embed_documents(texts: list[str]) -> list[list[float]]`는 **코퍼스 적재**에 쓴다.
두 메서드는 동일한 모델을 호출하지만, 배치 최적화 여부가 다르다.

In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

q_vec = embeddings.embed_query("LangChain은 무엇인가?")
print("query dim:", len(q_vec), "| preview:", q_vec[:4])

docs = [
    "LangChain은 LLM 애플리케이션을 조립하는 프레임워크다.",
    "RAG는 검색 기반 생성으로 환각을 줄인다.",
    "임베딩 벡터는 의미를 수치화한 고차원 표현이다.",
]
doc_vecs = embeddings.embed_documents(docs)
print("docs:", len(doc_vecs), "x", len(doc_vecs[0]))

## 2.01.3 차원 · 비용 · 다국어 성능

| 모델 | 기본 차원 | MTEB 영어 | 한국어(KLUE STS 근사) | 1K 토큰당 가격(2026-04 기준) |
|------|----------|-----------|------------------------|------------------------------|
| `text-embedding-3-small` | 1536 | 62.3 | 보통 | $0.00002 |
| `text-embedding-3-large` | 3072 | 64.6 | 양호 | $0.00013 |
| `text-embedding-ada-002` (레거시) | 1536 | 60.9 | 보통 | $0.00010 |

### 차원 축소 (`dimensions`)

`text-embedding-3-*` 계열은 Matryoshka representation을 지원한다. 원래 차원을 유지한 채 앞쪽 N차원만 잘라도 의미가 잘 보존된다.
vector DB 스키마를 작게 잡아 저장·검색 비용을 줄일 수 있다.

In [ ]:
reduced = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=512)
v = reduced.embed_query("차원을 512로 줄인 임베딩")
print("reduced dim:", len(v))

large = OpenAIEmbeddings(model="text-embedding-3-large")
v2 = large.embed_query("3-large는 기본 3072차원이다")
print("large dim:", len(v2))

## 2.01.4 벡터스토어 연동 미리보기

임베더는 vector store에 **주입해서** 쓰는 것이 일반적이다. 아래는 `Chroma` 인메모리 예시(자세한 건 `03_vectorstores`에서 다룸).

In [ ]:
# !pip install -U langchain-chroma
from langchain_chroma import Chroma

store = Chroma.from_texts(
    texts=docs,
    embedding=OpenAIEmbeddings(model="text-embedding-3-small"),
    collection_name="demo_openai",
)
hits = store.similarity_search("환각을 줄이는 방법", k=2)
for h in hits:
    print("-", h.page_content)

## 2.01.5 배치 튜닝과 긴 문서 처리

- `chunk_size`(기본 1000): `embed_documents` 호출을 몇 개씩 묶어 API에 보낼지 결정. 네트워크·rate limit에 맞춰 조정.
- `embedding_ctx_length`(기본 8191): 토큰 한도. 한도를 넘는 단일 문서는 `check_embedding_ctx_length=True`일 때 자동 청크되어 **평균** 벡터로 반환.
- `show_progress_bar=True`로 대량 적재 시 진행 상황 확인.

In [ ]:
bulk = OpenAIEmbeddings(
    model="text-embedding-3-small",
    chunk_size=200,            # 한 번에 200개씩 API 호출
    show_progress_bar=False,   # 노트북 실행 로그 간결화
)
vs = bulk.embed_documents([f"샘플 문장 {i}" for i in range(25)])
print("batch embedded:", len(vs), "vectors")

## 2.01.6 Azure OpenAI Embeddings

Azure 배포에서는 **배포 이름(deployment name)** 이 모델 ID를 대체한다. 네 가지 필수 값을 모두 주입한다.

| 필드 | 예시 | 설명 |
|------|------|------|
| `azure_deployment` | `my-embed-3-small` | Azure 포털에서 만든 배포 이름 |
| `openai_api_version` | `2024-02-01` | 배포 시점 기준 API 버전 |
| `azure_endpoint` | `https://<resource>.openai.azure.com/` | 리소스 엔드포인트 |
| `api_key` | `AZURE_OPENAI_API_KEY` 환경변수 | 또는 `AZURE_AD_TOKEN`으로 AAD 인증 |

In [ ]:
# Azure 배포가 구성되어 있다면 주석 해제
# from langchain_openai import AzureOpenAIEmbeddings
#
# azure_emb = AzureOpenAIEmbeddings(
#     azure_deployment=os.environ["AZURE_OPENAI_EMBED_DEPLOYMENT"],
#     openai_api_version="2024-02-01",
#     azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
#     api_key=os.environ["AZURE_OPENAI_API_KEY"],
# )
# print(azure_emb.embed_query("Azure 배포 테스트")[:3])
print("Azure 섹션은 예시 코드. 배포가 있다면 주석 해제 후 실행하라.")

## 체크리스트

- [ ] 쿼리는 `embed_query`, 코퍼스는 `embed_documents`로 나눠 부르는지
- [ ] vector store 스키마 차원이 `dimensions`와 일치하는지 (후에 바꾸면 전체 재적재)
- [ ] 대량 적재 시 `chunk_size`를 조정해 rate limit을 회피하는지
- [ ] Azure 경로는 4개 필드(`azure_deployment`, `openai_api_version`, `azure_endpoint`, `api_key`) 전부 명시

## 다음

- `02_google.ipynb` — Gemini Embedding과 `task_type` 파라미터
- `../03_vectorstores/` — 임베더를 Chroma/FAISS/Pinecone에 붙이기

## 참고

- 공식: https://docs.langchain.com/oss/python/integrations/text_embedding/openai
- OpenAI Embeddings 문서: https://platform.openai.com/docs/guides/embeddings
- Azure OpenAI: https://learn.microsoft.com/azure/ai-services/openai/concepts/models#embeddings